# Bio-Synthetica Pro — GRPO Training (Kaggle)

**OpenEnv Hackathon India 2026** | Llama-3.1-8B 4-bit + GRPO | T4 GPU

GitHub: https://github.com/Prantik-07/bio-synthetica

In [ ]:
# Step 1 — Install
import subprocess, sys
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-U',
    'pip', 'transformers', 'accelerate', 'bitsandbytes', 'peft', 'trl'], check=True)
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-U', 'unsloth'], check=True)
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q',
    'openenv', 'wandb', 'datasets', 'matplotlib', 'numpy'], check=True)
print('✅ Packages installed')

In [ ]:
# Step 2 — Verify GPU
import torch
assert torch.cuda.is_available(), 'No GPU! Go to Settings → Accelerator → GPU T4 x2'
print('GPU:', torch.cuda.get_device_name(0))
print('VRAM:', round(torch.cuda.get_device_properties(0).total_memory/1e9, 1), 'GB')

In [ ]:
# Step 3 — Inline environment code (no git clone needed)
import random, ast, re

WELL_IDS = ['A1','A2','A3','A4','B1','B2','B3','B4',
            'C1','C2','C3','C4','D1','D2','D3','D4']
MAX_WELL_VOLUME = 200
MAX_PIPETTE_VOLUME = 200
MIN_TEMP, MAX_TEMP = 4, 95
STARTING_BUDGET = 10.0
REAGENT_PRICES = {'buffer': 0.1, 'enzyme': 2.5, 'substrate': 1.0, 'water': 0.05}


class LabSimulator:
    def __init__(self, seed=None):
        self._seed = seed
        self._rng = random.Random(seed)
        self._init_state()

    def _init_state(self):
        chemicals = list(REAGENT_PRICES.keys())
        self.wells = {w: {'volume': round(self._rng.uniform(0, 100), 2),
                          'chemical': self._rng.choice(chemicals),
                          'temperature': 25.0, 'contaminated': False}
                      for w in WELL_IDS}
        self.tips_remaining = 8
        self.budget_remaining = STARTING_BUDGET
        self.contaminated_wells = set()
        self.scanned_wells = set()
        self.event_log = []
        self.step_count = 0

    def reset(self, seed=None):
        if seed is not None:
            self._seed = seed
            self._rng = random.Random(seed)
        else:
            self._rng = random.Random(self._seed)
        self._init_state()
        return self.get_full_state()

    def scan(self, well_id):
        if well_id not in WELL_IDS:
            return {'success': False, 'error': f'Unknown well: {well_id}'}
        self.scanned_wells.add(well_id)
        w = self.wells[well_id]
        return {'success': True, 'well_id': well_id, 'volume': w['volume'],
                'chemical': w['chemical'], 'temperature': w['temperature'],
                'contaminated': w['contaminated']}

    def pipette(self, source, dest, volume):
        v = []
        if source not in self.scanned_wells: v.append(f'{source} not scanned')
        if dest not in self.scanned_wells: v.append(f'{dest} not scanned')
        if volume > MAX_PIPETTE_VOLUME: v.append(f'Volume {volume}ul exceeds max')
        if source in WELL_IDS and volume > self.wells[source]['volume']:
            v.append(f'Insufficient volume in {source}')
        if dest in WELL_IDS and self.wells[dest]['volume'] + volume > MAX_WELL_VOLUME:
            v.append(f'Would overflow {dest}')
        if source in self.contaminated_wells: v.append(f'{source} contaminated')
        if dest in self.contaminated_wells: v.append(f'{dest} contaminated')
        if v: return {'success': False, 'violations': v, 'cost': 0}
        chem = self.wells[source]['chemical']
        cost = round(volume * REAGENT_PRICES[chem], 4)
        self.wells[source]['volume'] = round(self.wells[source]['volume'] - volume, 2)
        self.wells[dest]['volume'] = round(self.wells[dest]['volume'] + volume, 2)
        self.wells[dest]['chemical'] = chem
        self.budget_remaining = round(self.budget_remaining - cost, 4)
        self.step_count += 1
        return {'success': True, 'violations': [], 'cost': cost}

    def mix(self, well_id, volume, repetitions):
        v = []
        if well_id not in self.scanned_wells: v.append(f'{well_id} not scanned')
        if well_id in self.contaminated_wells: v.append(f'{well_id} contaminated')
        if well_id in WELL_IDS and volume > self.wells[well_id]['volume']:
            v.append('Mix volume exceeds well volume')
        if v: return {'success': False, 'violations': v, 'cost': 0}
        cost = round(0.1 * repetitions, 4)
        self.budget_remaining = round(self.budget_remaining - cost, 4)
        self.step_count += 1
        return {'success': True, 'violations': [], 'cost': cost}

    def set_temperature(self, well_id, temp):
        v = []
        if well_id not in self.scanned_wells: v.append(f'{well_id} not scanned')
        if temp < MIN_TEMP or temp > MAX_TEMP: v.append(f'Temp {temp}C out of range')
        if v: return {'success': False, 'violations': v, 'cost': 0}
        self.wells[well_id]['temperature'] = temp
        self.budget_remaining = round(self.budget_remaining - 0.2, 4)
        self.step_count += 1
        return {'success': True, 'violations': [], 'cost': 0.2}

    def aspirate(self, well_id, volume):
        v = []
        if well_id not in self.scanned_wells: v.append(f'{well_id} not scanned')
        if volume > MAX_PIPETTE_VOLUME: v.append('Volume exceeds max')
        if well_id in WELL_IDS and volume > self.wells[well_id]['volume']:
            v.append('Insufficient volume')
        if well_id in self.contaminated_wells: v.append(f'{well_id} contaminated')
        if v: return {'success': False, 'violations': v, 'cost': 0}
        cost = round(volume * 0.01, 4)
        self.wells[well_id]['volume'] = round(self.wells[well_id]['volume'] - volume, 2)
        self.budget_remaining = round(self.budget_remaining - cost, 4)
        self.step_count += 1
        return {'success': True, 'violations': [], 'cost': cost}

    def dispense(self, well_id, volume):
        v = []
        if well_id not in self.scanned_wells: v.append(f'{well_id} not scanned')
        if well_id in WELL_IDS and self.wells[well_id]['volume'] + volume > MAX_WELL_VOLUME:
            v.append('Would overflow')
        if well_id in self.contaminated_wells: v.append(f'{well_id} contaminated')
        if v: return {'success': False, 'violations': v, 'cost': 0}
        self.wells[well_id]['volume'] = round(self.wells[well_id]['volume'] + volume, 2)
        self.step_count += 1
        return {'success': True, 'violations': [], 'cost': 0}

    def discard_tip(self):
        if self.tips_remaining == 0:
            return {'success': False, 'error': 'No tips remaining'}
        self.tips_remaining -= 1
        self.step_count += 1
        return {'success': True, 'tips_remaining': self.tips_remaining}

    def trigger_contamination_event(self, well_id):
        self.contaminated_wells.add(well_id)
        self.wells[well_id]['contaminated'] = True
        ev = {'event': 'contamination_alert', 'affected_well': well_id,
              'message': f'ALERT: Well {well_id} is contaminated. Avoid it.'}
        self.event_log.append(ev)
        return ev

    def check_goal(self, goal):
        tw, tc = goal.get('target_well'), goal.get('target_chemical')
        tconc = goal.get('target_concentration', 0.5)
        tol = goal.get('tolerance', 0.05)
        if tw not in WELL_IDS: return 0.0
        w = self.wells[tw]
        if w['chemical'] != tc: return 0.0
        conc = w['volume'] / MAX_WELL_VOLUME
        diff = abs(conc - tconc)
        if diff <= tol: return 1.0
        elif diff <= 2*tol: return 0.5
        return round(max(0.0, 1.0 - diff/1.0) * 0.4, 4)

    def get_full_state(self):
        return {'wells': {wid: {**d, 'scanned': wid in self.scanned_wells}
                          for wid, d in self.wells.items()},
                'pipette': {'max_volume': MAX_PIPETTE_VOLUME,
                            'tips_remaining': self.tips_remaining},
                'budget_remaining': self.budget_remaining,
                'contaminated_wells': list(self.contaminated_wells),
                'scanned_wells': list(self.scanned_wells),
                'step_count': self.step_count,
                'event_log': self.event_log}


class RewardCalculator:
    def compute(self, action_result, state):
        if not action_result.get('syntax_pass', False):
            return -0.5
        r = 0.1
        n = len(action_result.get('violations', []))
        r += 0.3 if n == 0 else -min(0.15*n, 0.45)
        r += 1.0 * action_result.get('goal_progress', 0.0)
        steps = state.get('episode_step', 0)
        r += 0.3 if steps < 8 else (0.1 if steps < 12 else 0.0)
        r += 0.3 * max(0.0, state.get('budget_remaining', 10.0) / 10.0)
        if (state.get('event') == 'contamination_alert'
                and action_result.get('rerouted_successfully', False)):
            r += 0.5
        return round(r, 4)


def normalize_protocol(text):
    if not text: return ''
    s = str(text).strip()
    for prefix in ('Here is the protocol','Here is the code',
                   "Here's the protocol",'Sure,','Sure!'):
        if s.lower().startswith(prefix.lower()):
            idx = s.find('\n')
            s = s[idx+1:].lstrip() if idx != -1 else ''
            break
    if '```' in s:
        parts = re.split(r'```(?:\s*python)?\s*', s, maxsplit=1, flags=re.IGNORECASE)
        if len(parts) > 1:
            rest = parts[1]
            end = rest.find('```')
            s = rest[:end] if end != -1 else rest
    s = s.strip()
    if s.lower().startswith('python\n'): s = s[7:].lstrip()
    return s.strip()


def _actions_from_stmts(stmts):
    actions = []
    for stmt in stmts:
        if isinstance(stmt, ast.Expr) and isinstance(stmt.value, ast.Call):
            call = stmt.value
            if isinstance(call.func, ast.Name):
                args = []
                for a in call.args:
                    try: args.append(ast.literal_eval(a))
                    except: args.append(None)
                for kw in call.keywords:
                    try: args.append(ast.literal_eval(kw.value))
                    except: args.append(None)
                actions.append((call.func.id, args))
    return actions


class BioSyntheticaEnv:
    def __init__(self):
        self.simulator = LabSimulator()
        self.episode_step = 0
        self.max_steps = 15
        self.contamination_triggered = False
        self.active_event = None
        self.current_goal = None

    def _make_obs(self, goal, event=None):
        state = self.simulator.get_full_state()
        wells = state['wells']
        scanned = set(state['scanned_wells'])
        unscanned = [w for w in WELL_IDS if w not in scanned]
        scanned_lines = []
        for wid in sorted(scanned):
            w = wells[wid]
            tag = '  ⚠️ CONTAMINATED' if w['contaminated'] else ''
            noisy = round(w['volume'] * (1 + random.gauss(0, 0.05)), 2)
            scanned_lines.append(f"  {wid}: volume={noisy}ul, chemical={w['chemical']}, temp={w['temperature']}C{tag}")
        scanned_block = '\n'.join(scanned_lines) if scanned_lines else '  (none scanned yet)'
        hidden_block = ', '.join(unscanned) if unscanned else '(all wells scanned)'
        alert = f"\nACTIVE ALERT: {event['message']}\n" if event else ''
        return f"""=== BIO-SYNTHETICA LAB STATE ===
SCANNED WELLS (you can use these):\n{scanned_block}
HIDDEN WELLS (scan before using): {hidden_block}
BUDGET: ${state['budget_remaining']:.2f}
GOAL: {goal['description']}{alert}
ACTIONS: scan(well_id), pipette(src, dst, vol), mix(well_id, vol, reps),
  set_temperature(well_id, temp), aspirate(well_id, vol), dispense(well_id, vol),
  discard_tip(), report_complete()
Output ONLY Python code."""

    def reset(self):
        self.simulator.reset()
        tw = random.choice(WELL_IDS)
        tc = random.choice(list(REAGENT_PRICES.keys()))
        tconc = round(random.uniform(0.3, 0.8), 2)
        self.current_goal = {'target_well': tw, 'target_chemical': tc,
                             'target_concentration': tconc, 'tolerance': 0.05,
                             'description': f'Achieve {tconc}x conc of {tc} in well {tw}'}
        self.episode_step = 0
        self.contamination_triggered = False
        self.active_event = None
        return {'observation': self._make_obs(self.current_goal), 'goal': self.current_goal,
                'step': 0, 'done': False, 'info': {}}

    def step(self, action):
        self.episode_step += 1
        action = normalize_protocol(action)
        if (not self.contamination_triggered and
                3 <= self.episode_step <= 7 and random.random() < 0.3):
            target = random.choice(WELL_IDS)
            self.active_event = self.simulator.trigger_contamination_event(target)
            self.contamination_triggered = True
        # Parse
        try:
            tree = ast.parse(action) if action.strip() else None
        except SyntaxError:
            tree = None
        if tree is None or not tree.body:
            syntax_pass, actions_list = False, []
        else:
            if len(tree.body) == 1 and isinstance(tree.body[0], (ast.FunctionDef, ast.AsyncFunctionDef)):
                stmts = tree.body[0].body
            else:
                stmts = tree.body
            actions_list = _actions_from_stmts(stmts)
            syntax_pass = bool(actions_list)
        routing = {
            'scan': lambda a: self.simulator.scan(*a),
            'pipette': lambda a: self.simulator.pipette(*a),
            'mix': lambda a: self.simulator.mix(*a),
            'set_temperature': lambda a: self.simulator.set_temperature(*a),
            'aspirate': lambda a: self.simulator.aspirate(*a),
            'dispense': lambda a: self.simulator.dispense(*a),
            'discard_tip': lambda a: self.simulator.discard_tip(),
            'report_complete': lambda a: {'success': True, 'violations': []},
        }
        all_violations, goal_progress, rerouted = [], 0.0, False
        if syntax_pass:
            for aname, args in actions_list:
                res = routing.get(aname, lambda a: {'success':False,'violations':[f'Unknown: {aname}']})(args)
                all_violations.extend(res.get('violations', []))
                if aname == 'report_complete': break
            goal_progress = self.simulator.check_goal(self.current_goal)
            if self.contamination_triggered:
                used = {args[0] for aname, args in actions_list if args}
                if not self.simulator.contaminated_wells.intersection(used):
                    rerouted = True
        info = {'violations': all_violations, 'goal_progress': goal_progress,
                'budget_used': round(10.0 - self.simulator.budget_remaining, 4),
                'steps_taken': self.episode_step,
                'event_triggered': self.contamination_triggered,
                'rerouted_successfully': rerouted,
                'syntax_pass': syntax_pass}
        state = self.simulator.get_full_state()
        state['episode_step'] = self.episode_step
        state['budget_remaining'] = self.simulator.budget_remaining
        state['event'] = 'contamination_alert' if self.contamination_triggered else None
        reward = RewardCalculator().compute(info, state)
        done = self.episode_step >= self.max_steps or 'report_complete' in (action or '')
        return ({'observation': self._make_obs(self.current_goal, self.active_event),
                 'goal': self.current_goal, 'step': self.episode_step,
                 'done': done, 'info': info}, reward, done, info)


# Quick sanity check
env = BioSyntheticaEnv()
obs = env.reset()
print(obs['observation'][:400])
_, r, _, info = env.step('scan("A1")\nscan("B1")\npipette("A1","B1",50)\nreport_complete()')
print(f'\nReward: {r} | syntax_pass: {info["syntax_pass"]}')
print('✅ Environment OK')

In [ ]:
# Step 4 — WandB (optional)
import os
WANDB_KEY = ''  # paste your key here, or leave blank to skip
import wandb
if WANDB_KEY:
    wandb.login(key=WANDB_KEY)
else:
    os.environ['WANDB_DISABLED'] = 'true'
    print('WandB disabled — training still runs, plots saved locally')

In [ ]:
# Step 5 — Load model
import gc, os, torch
os.environ.setdefault('PYTORCH_CUDA_ALLOC_CONF', 'expandable_segments:True')
torch.cuda.set_device(0); torch.cuda.empty_cache(); gc.collect()

from unsloth import FastLanguageModel, PatchFastRL
PatchFastRL('GRPO', FastLanguageModel)

max_seq_length = 1024

def _load(name):
    return FastLanguageModel.from_pretrained(
        model_name=name, max_seq_length=max_seq_length,
        load_in_4bit=True, dtype=torch.float16,
        fast_inference=False, device_map={'': 0})

PRIMARY = 'unsloth/Meta-Llama-3.1-8B-Instruct-bnb-4bit'
FALLBACK = 'unsloth/Llama-3.2-3B-Instruct-bnb-4bit'
model_name_used = PRIMARY
try:
    model, tokenizer = _load(PRIMARY)
except ValueError as e:
    if 'cpu' in str(e).lower() or 'disk' in str(e).lower():
        print('⚠️ 8B failed — loading 3.2-3B fallback (same GRPO code)')
        model_name_used = FALLBACK
        model, tokenizer = _load(FALLBACK)
    else:
        raise

model = FastLanguageModel.get_peft_model(
    model, r=16,
    target_modules=['q_proj','k_proj','v_proj','o_proj',
                    'gate_proj','up_proj','down_proj'],
    lora_alpha=16, lora_dropout=0, bias='none',
    use_gradient_checkpointing='unsloth', random_state=42)
print('✅ Model loaded:', model_name_used)

In [ ]:
# Step 6 — Dataset & reward
from datasets import Dataset

SYSTEM_PROMPT = """You are a lab automation scientist operating an Opentrons OT-2 robot.
RULES:
1. Always call scan(well_id) before using any well
2. Never exceed 200ul volume in any well
3. Never pipette more than 200ul at once
4. Temperature must stay between 4C and 95C only
5. Never use contaminated wells
6. Minimize reagent cost
7. If contamination alert fires, avoid that well
8. Always end with report_complete()
OUTPUT ONLY VALID PYTHON CODE. No markdown. No comments."""


def make_dataset(n=200):
    e = BioSyntheticaEnv()
    rows = []
    for _ in range(n):
        o = e.reset()
        rows.append({'prompt': f"{SYSTEM_PROMPT}\n\n{o['observation']}", 'completion': ''})
    return Dataset.from_list(rows)


def reward_fn(completions, prompts=None, **_):
    e = BioSyntheticaEnv()
    rewards = []
    for c in completions:
        try:
            e.reset()
            _, r, _, _ = e.step(c)
            rewards.append(r)
        except Exception:
            rewards.append(-0.5)
    return rewards


dataset = make_dataset(200)
print('Dataset size:', len(dataset))
print('Sample (first 300 chars):')
print(dataset[0]['prompt'][:300])

In [ ]:
# Step 7 — GRPO config & train
import os
import warnings
# Noisy but harmless: TRL uses max_new_tokens=512; model config still has max_length=131072 — 512 wins.
warnings.filterwarnings('ignore', message=r'Both .*max_new_tokens.*max_length.*')
warnings.filterwarnings('ignore', category=FutureWarning, module=r'transformers\.modeling_attn_mask_utils')
from trl import GRPOTrainer, GRPOConfig

grpo_config = GRPOConfig(
    learning_rate=2e-5,
    per_device_train_batch_size=2,
    num_generations=4,
    max_completion_length=512,
    max_steps=200,
    logging_steps=10,
    save_steps=50,
    output_dir='/kaggle/working/bio-synthetica-checkpoints',
    report_to='wandb' if os.environ.get('WANDB_DISABLED') != 'true' else 'none',
    warmup_steps=20,
    weight_decay=0.01,
)
grpo_config.max_completion_length = 512
print('max_completion_length =', grpo_config.max_completion_length)

trainer = GRPOTrainer(
    model=model,
    args=grpo_config,
    reward_funcs=reward_fn,
    train_dataset=dataset,
    tokenizer=tokenizer,
)
print('trainer.max_completion_length =', getattr(trainer, 'max_completion_length', 'n/a'))
print('🚀 Starting GRPO training...')
trainer.train()
print('✅ Training complete')

In [ ]:
# Step 8 — Evaluate
from unsloth import FastLanguageModel
FastLanguageModel.for_inference(model)

env_eval = BioSyntheticaEnv()
rewards_eval = []
for i in range(10):
    obs = env_eval.reset()
    prompt = f"{SYSTEM_PROMPT}\n\n{obs['observation']}"
    inputs = tokenizer(prompt, return_tensors='pt').to('cuda')
    with torch.no_grad():
        out = model.generate(**inputs, max_new_tokens=256, temperature=0.7,
                              do_sample=True, pad_token_id=tokenizer.eos_token_id)
    generated = tokenizer.decode(out[0][inputs['input_ids'].shape[1]:], skip_special_tokens=True)
    _, r, _, info = env_eval.step(generated)
    rewards_eval.append(r)
    print(f'  Eval {i+1}: reward={r:.3f} | syntax={info["syntax_pass"]} | violations={len(info["violations"])}')

print(f'\nMean eval reward: {sum(rewards_eval)/len(rewards_eval):.3f}')

In [ ]:
# Step 9 — Save
SAVE_PATH = '/kaggle/working/bio-synthetica-final'
model.save_pretrained(SAVE_PATH)
tokenizer.save_pretrained(SAVE_PATH)
print('✅ Model saved to', SAVE_PATH)